# Leiden annotation

In [ ]:
import scvi
import scanpy as sc

import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt

import sys
import os

## Root directory

In [ ]:
os.chdir('/research/peer/fdeckert/FD20250213HSCT')

## Custom modules

In [ ]:
sys.path.append('bin/')
from adata_qc import *

## Settup rpy2 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.4.1-FD20250213HSCT/lib/R'

In [ ]:
import rpy2
%load_ext rpy2.ipython

## Figures 

In [ ]:
sc.set_figure_params(figsize=(7.5, 7.5), dpi_save=1200, fontsize=16, frameon=False, facecolor='white')
mpl.rcParams['figure.facecolor'] = 'white'

# Import AnnData

In [ ]:
adata = sc.read_h5ad('data/object/qc.h5ad')

# Filter doublet cell and cluster

In [ ]:
sc.pl.umap(adata, color=['leiden', 'solo_pred_doublet', 'solo_label', 'hto_demux_class', 'celltypist_label_high', 'log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_MT', 'pct_counts_RP', 'pct_counts_HB'], frameon=False, ncols=5, wspace=0.25, size=20, legend_loc='on data', use_raw=False)

In [ ]:
adata = adata[(adata.obs.hto_demux_class.isin(['Blood', 'Skin']))]

In [ ]:
sc.pl.umap(adata, color=['leiden', 'solo_pred_doublet', 'solo_label', 'hto_demux_class', 'celltypist_label_high', 'log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_MT', 'pct_counts_RP', 'pct_counts_HB'], frameon=False, ncols=5, wspace=0.25, size=20, legend_loc='on data', use_raw=False)

# Cell type marker genes

In [ ]:
celltype_genes_dict = {
    
    r'Immune cell': ['PTPRC'], 
    r'Fibroblast': ['THY1', 'MGP', 'MFAP5'], 
    r'F1': ['COL13A1', 'COL18A1', 'COL23A1', 'APCDD1', 'WIF1', 'NKD2'], 
    r'F2': ['PI16', 'CD34', 'DPP4', 'FGF18', 'CNN3'], 
    r'F2_3': ['PPARG','SFRP1','PLA2G2A','APOE','C7'], 
    r'F3': ['CCL19', 'CXCL12', 'CH25H', 'IL33', 'IL15', 'TNFSF13B', 'VCAM1', 'CD74'], 
    r'F4': [ 'TNN', 'COCH', 'CRABP1', 'MKX', 'TNMD'], # Not found RSP04
    r'F4 DS': ['MEF2C', 'MYL4'], # Not found: DEP1
    r'F4 DP': [ 'CORIN', 'HHIP', 'LEF1'], # Not found RSP03
    r'Keratinocyte': ['KRT5', 'KRT14', 'KRT1', 'KRT10'], 
    r'Endothelial cell': ['PECAM1', 'CD74', 'LYVE1', 'CCL21'], 
    r'Pericyte': ['TAGLN', 'ACTA2', 'CCL19', 'CCL26'], 
    r'Melanocyte': ['PMEL', 'MLANA'], 
    
    r'$TRC\alpha$': [i for i in adata.var_names if 'TRAC' in i],
    r'$TRC\beta$': [i for i in adata.var_names if 'TRBC' in i],
    r'$TRC\gamma$': [i for i in adata.var_names if 'TRGC' in i],
    r'$TRC\delta$': [i for i in adata.var_names if 'TRDC' in i],
    r'CD3': ['CD3E', 'CD3D', 'CD3G', 'CD247'],
    r'CD4': ['CD4'], 
    r'CD8': ['CD8A', 'CD8B'],
    r'Regulatory': ['CTLA4', 'IL2RA', 'FOXP3', 'TIGIT'], 
    r'Cytotoxic': ['KLRB1', 'GZMB', 'CCL5', 'NKG7', 'PRF1', 'GZMM'], # CD161:KLRB1
    r'Type 17': ['CCR6', 'RORC'], 
    r'NK': ['NCAM1', 'KLRD1', 'FCGR3A'], # CD56:NCAM1, CD94:KLRD1, CD16:FCGR3A | Possible TRGC/TRDC | CD3E has been found
    r'$T_{inv}$': ['ZBTB16'], # PZLF:ZBTB16, CD4+ or DN, four classes with some having restriceted TRAC/TRBC | Marker overlap with NK | Type I T_{inv} Vα24Jα18
    r'MAIT': ['SLC4A10', 'CEBPD'], # Vα7.2Jα33, TRAV1-2
    r'Residency': ['CREM', 'LMNA', 'EZR', 'ZFP36', 'CD69', 'CD27', 'ITGAE'], # CD69 distinguishes memory T cell in tissue from blood
    r'Naive': ['TCF7', 'SELL', 'CCR7'], # CD127:IL7R, CD62L:SELL
    r'Experienced': ['CD44'],  
        
    r'B cell': ['MS4A1', 'CD19', 'IGHD', 'IGHA1'], 
    r'Monocytes': ['CD14', 'FCGR3A', 'CCR2', 'CX3CR1'], 
    r'Macrophages': ['CD163', 'MRC1', 'F13A1'], 
    r'cDC1': ['XCR1', 'THBD', 'CLEC9A'], 
    r'cDC2': ['CD1C', 'SIRPA', 'CLEC10A'], 
    r'LC': ['CD1A', 'CD207'], 
    r'pDC': ['IL3RA', 'TCF4', 'CLEC4C'],  
    r'Migration': ['CCR7'], 
    r'Maturation': ['LAMP3', 'BIRC3'], 
    r'HSC': ['CD34'], 
    r'Mast cell': ['KIT', 'MS4A2'], 
    
    r'Cycling': ['PCNA', 'MKI67', 'TOP2A', 'CDK1'], 
    
}

# Blood subset

In [ ]:
adata_0 = adata[adata.obs.hto_demux_class=='Blood']
adata_0 = adata_0[:, (adata_0.X>=3).sum(axis=0)>=3]

In [ ]:
adata_0 = adata_0.copy()

In [ ]:
cache_scvi = True

In [ ]:
if not cache_scvi:
    
    scvi.model.SCVI.setup_anndata(
    
        adata_0,
        batch_key='sample_name', 
        continuous_covariate_keys=['S_ucell', 'G2M_ucell', 'pct_counts_MT']
        
    )
    
    model = scvi.model.SCVI(
    
        adata_0, 
        n_latent=30, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'

    )
    
    max_epochs = int(np.min([round((20000 / adata_0.n_obs) * 400), 400]))
    
    model.train(max_epochs=max_epochs, check_val_every_n_epoch=1)
    
    model.save('data/object/scvi/annotation/model_0/', overwrite=True)
    
else: 
    
    model = scvi.model.SCVI.load('data/object/scvi/annotation/model_0/', adata=adata_0)

In [ ]:
cache_dim = True

In [ ]:
if not cache_dim: 
    
    adata_0.obsm['latent'] = model.get_latent_representation()
    
    sc.pp.neighbors(adata_0, n_neighbors=30, use_rep='latent')
    sc.tl.leiden(adata_0, resolution=1, flavor='igraph', n_iterations=2, key_added='leiden')
    sc.tl.umap(adata_0, min_dist=1)
    
    adata_0.write_h5ad('data/object/scvi/annotation/adata_0.h5ad')
    
else: 
    
    adata_0 = sc.read_h5ad('data/object/scvi/annotation/adata_0.h5ad')

In [ ]:
adata_0 = adata_0.raw.to_adata()
adata_0.raw = adata_0.copy()

In [ ]:
sc.pp.normalize_total(adata_0, target_sum=10000)
sc.pp.log1p(adata_0)

In [ ]:
# Filter celltypist annotation 
adata_0.obs['celltypist_label_low'] = adata_0.obs['celltypist_label_low'].to_list()
adata_0.obs = adata_0.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata_0.n_obs/100), 'freq < 1 %') if x.name=='celltypist_label_low' else x)

adata_0.obs['celltypist_label_high'] = adata_0.obs['celltypist_label_high'].to_list()
adata_0.obs = adata_0.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata_0.n_obs/100), 'freq < 1 %') if x.name=='celltypist_label_high' else x)

In [ ]:
sc.tl.leiden(adata_0, resolution=0.50, restrict_to=('leiden', ['17']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')
sc.tl.leiden(adata_0, resolution=0.50, restrict_to=('leiden_restrict', ['5']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')
sc.tl.leiden(adata_0, resolution=0.50, restrict_to=('leiden_restrict', ['30']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')

adata_0.obs['leiden'] = adata_0.obs['leiden_restrict']
del adata_0.obs['leiden_restrict']

In [ ]:
celltype_leiden = {
    
    '0': r'$NK\ CD16+\ (0)$',
    '1': r'$NK\ CD16-\ (1)$',
    '2': r'$T_{c}\ E/EM\ (2)$',
    '3': r'$T_{c}\ E/EM\ (3)$',
    '4': r'$T_{inv}\ (4)$',
    '5,0': r'$T_{h}\ E/EM\ (5,0)$',
    '5,1': r'$T_{c}\ E/EM\ (5,1)$',
    '5,2': r'$T_{h}\ E/EM\ (5,2)$',
    '5,3': r'$T_{h}\ E/EM\ (5,3)$',
    '6': r'$Mono_{c}\ (6)$',
    '7': r'$Mono_{int}\ (7)$',
    '8': r'$Mono_{c}\ (8)$',
    '9': r'$Mono_{c}\ (9)$',
    '10': r'$Artefact\ (10)$',
    '11': r'$Mono_{c}\ (11)$',
    '12': r'$T_{h}\ E/EM\ (12)$',
    '13': r'$cDC2\ (13)$',
    '14': r'$Artefact\ (14)$', 
    '15': r'$Mono_{nc}\ (15)$',
    '16': r'$T_{c}\ E/EM\ (16)$',
    '17,0': r'$T_{h}\ N/CM\ (17,0)$',
    '17,1': r'$T_{h}\ N/CM\ (17,1)$',
    '17,2': r'$T_{c}\ N/CM\ (17,2)$',
    '17,3': r'$T_{h}\ N/CM\ (17,3)$',
    '18': r'$Artefact\ (18)$',
    '19': r'$Artefact\ (19)$',
    '20': r'$Artefact\ (20)$', 
    '21': r'$Artefact\ (21)$', 
    '22': r'$T_{reg}\ (22)$', 
    '23': r'$HSC/MPP\ (23)$',
    '24': r'$Mono_{c}\ (24)$', 
    '25': r'$T_{c}\ E/EM\ (25)$', 
    '26': r'$Artefact\ (26)$', 
    '27': r'$pDC\ (27)$', 
    '28': r'$T_{c}\ E/EM\ (28)$', 
    '29': r'$Plasma\ cell\ (29)$', 
    '30,0': r'$B\ cell\ (30,0)$', 
    '30,1': r'$B\ cell\ (30,1)$', 
    '30,2': r'$B\ cell\ (30,2)$', 
    '30,3': r'$Artefact\ (30,3)$'
    
} 

In [ ]:
# Mapper 
celltype_leiden = pd.DataFrame({'celltype_leiden': adata_0.obs['leiden'].map(lambda x: celltype_leiden.get(x, x)).astype('category')})
adata_0.obs = adata_0.obs.merge(celltype_leiden, left_index=True, right_index=True, how='left')

In [ ]:
adata_0.obs['celltype_leiden'] = adata_0.obs['celltype_leiden'].cat.reorder_categories([

    r'$Mono_{c}\ (9)$',
    r'$Mono_{c}\ (24)$',
    r'$Mono_{c}\ (6)$',
    r'$Mono_{c}\ (8)$',
    r'$Mono_{c}\ (11)$',
    
    r'$Mono_{int}\ (7)$',
    
    r'$Mono_{nc}\ (15)$',
    
    r'$cDC2\ (13)$',
    r'$pDC\ (27)$',
    
    r'$T_{h}\ N/CM\ (17,0)$',
    r'$T_{h}\ N/CM\ (17,1)$',
    r'$T_{h}\ N/CM\ (17,3)$',
    
    r'$T_{h}\ E/EM\ (12)$',
    r'$T_{h}\ E/EM\ (5,0)$',
    r'$T_{h}\ E/EM\ (5,2)$',
    r'$T_{h}\ E/EM\ (5,3)$',
    
    r'$T_{reg}\ (22)$', 

    r'$T_{c}\ N/CM\ (17,2)$',
    
    r'$T_{c}\ E/EM\ (2)$',
    r'$T_{c}\ E/EM\ (3)$',
    r'$T_{c}\ E/EM\ (5,1)$',
    r'$T_{c}\ E/EM\ (16)$',
    r'$T_{c}\ E/EM\ (25)$',
    r'$T_{c}\ E/EM\ (28)$',
    
    r'$T_{inv}\ (4)$',
    
    r'$NK\ CD16+\ (0)$',
    r'$NK\ CD16-\ (1)$',
    
    r'$B\ cell\ (30,0)$', 
    r'$B\ cell\ (30,1)$', 
    r'$B\ cell\ (30,2)$', 
    r'$Plasma\ cell\ (29)$',
    
    r'$HSC/MPP\ (23)$',
    
    r'$Artefact\ (10)$',
    r'$Artefact\ (14)$',
    r'$Artefact\ (18)$',
    r'$Artefact\ (19)$',
    r'$Artefact\ (20)$',
    r'$Artefact\ (21)$',
    r'$Artefact\ (26)$', 
    r'$Artefact\ (30,3)$'
    
])

In [ ]:
dp = sc.pl.dotplot(adata_0, celltype_genes_dict, groupby='celltype_leiden', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
celltype_low = {
    
    '0': r'$NK\ CD16+$',
    '1': r'$NK\ CD16-$',
    '2': r'$T_{c}\ E/EM$',
    '3': r'$T_{c}\ E/EM$',
    '4': r'$T_{inv}$',
    '5,0': r'$T_{h}\ E/EM$',
    '5,1': r'$T_{c}\ E/EM$',
    '5,2': r'$T_{h}\ E/EM$',
    '5,3': r'$T_{h}\ E/EM$',
    '6': r'$Mono_{c}$',
    '7': r'$Mono_{int}$',
    '8': r'$Mono_{c}$',
    '9': r'$Mono_{c}$',
    '10': r'$Artefact$',
    '11': r'$Mono_{c}$',
    '12': r'$T_{h}\ E/EM$',
    '13': r'$cDC2$',
    '14': r'$Artefact$', 
    '15': r'$Mono_{nc}$',
    '16': r'$T_{c}\ E/EM$',
    '17,0': r'$T_{h}\ N/CM$',
    '17,1': r'$T_{h}\ N/CM$',
    '17,2': r'$T_{c}\ N/CM$',
    '17,3': r'$T_{h}\ N/CM$',
    '18': r'$Artefact$',
    '19': r'$Artefact$',
    '20': r'$Artefact$', 
    '21': r'$Artefact$', 
    '22': r'$T_{reg}$', 
    '23': r'$HSC/MPP$',
    '24': r'$Mono_{c}$', 
    '25': r'$T_{c}\ E/EM$', 
    '26': r'$Artefact$', 
    '27': r'$pDC$', 
    '28': r'$T_{c}\ E/EM$', 
    '29': r'$Plasma\ cell$', 
    '30,0': r'$B\ cell$', 
    '30,1': r'$B\ cell$', 
    '30,2': r'$B\ cell$', 
    '30,3': r'$Artefact$'

}

In [ ]:
# Mapper 
celltype_low = pd.DataFrame({'celltype_low': adata_0.obs['leiden'].map(lambda x: celltype_low.get(x, x)).astype('category')})
adata_0.obs = adata_0.obs.merge(celltype_low, left_index=True, right_index=True, how='left')

In [ ]:
adata_0.obs['celltype_low'] = adata_0.obs['celltype_low'].cat.reorder_categories([
    
    r'$Mono_{c}$',
    r'$Mono_{int}$',
    r'$Mono_{nc}$',
    
    r'$cDC2$',
    r'$pDC$',
    
    r'$T_{h}\ N/CM$',
    r'$T_{h}\ E/EM$',
    
    r'$T_{reg}$',
    
    r'$T_{c}\ N/CM$',
    r'$T_{c}\ E/EM$',
    
    r'$T_{inv}$',
    
    r'$NK\ CD16-$',
    r'$NK\ CD16+$',
    
    r'$Plasma\ cell$',
    r'$B\ cell$', 
    
    r'$HSC/MPP$',
    r'$Artefact$'
    
])

In [ ]:
dp = sc.pl.dotplot(adata_0, celltype_genes_dict, groupby='celltype_low', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
celltype_high = {
    
    '0': r'$ILC$',
    '1': r'$ILC$',
    '2': r'$T\ cell$',
    '3': r'$T\ cell$',
    '4': r'$T\ cell$',
    '5,0': r'$T\ cell$',
    '5,1': r'$T\ cell$',
    '5,2': r'$T\ cell$',
    '5,3': r'$T\ cell$',
    '6': r'$Monocyte$',
    '7': r'$Monocyte$',
    '8': r'$Monocyte$',
    '9': r'$Monocyte$',
    '10': r'$Artefact$',
    '11': r'$Monocyte$',
    '12': r'$T\ cell$',
    '13': r'$cDC$',
    '14': r'$Artefact$', 
    '15': r'$Monocyte$',
    '16': r'$T\ cell$',
    '17,0': r'$T\ cell$',
    '17,1': r'$T\ cell$',
    '17,2': r'$T\ cell$',
    '17,3': r'$T\ cell$',
    '18': r'$Artefact$',
    '19': r'$Artefact$',
    '20': r'$Artefact$', 
    '21': r'$Artefact$', 
    '22': r'$T\ cell$', 
    '23': r'$HSC/MPP$',
    '24': r'$Monocyte$', 
    '25': r'$T\ cell$', 
    '26': r'$Artefact$', 
    '27': r'$pDC$', 
    '28': r'$T\ cell$', 
    '29': r'$B\ cell$', 
    '30,0': r'$B\ cell$', 
    '30,1': r'$B\ cell$', 
    '30,2': r'$B\ cell$', 
    '30,3': r'$Artefact$'

}

In [ ]:
# Mapper 
celltype_high = pd.DataFrame({'celltype_high': adata_0.obs['leiden'].map(lambda x: celltype_high.get(x, x)).astype('category')})
adata_0.obs = adata_0.obs.merge(celltype_high, left_index=True, right_index=True, how='left')

In [ ]:
adata_0.obs['celltype_high'] = adata_0.obs['celltype_high'].cat.reorder_categories([
    
    r'$Monocyte$',
    
    r'$cDC$',
    
    r'$pDC$',
    
    r'$T\ cell$',
    
    r'$ILC$',
    
    r'$B\ cell$', 

    r'$HSC/MPP$',

    r'$Artefact$'
    
])

In [ ]:
dp = sc.pl.dotplot(adata_0, celltype_genes_dict, groupby='celltype_high', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
celltype_domain = {
    
    '0': r'$Immune\ cell$',
    '1': r'$Immune\ cell$',
    '2': r'$Immune\ cell$',
    '3': r'$Immune\ cell$',
    '4': r'$Immune\ cell$',
    '5,0': r'$Immune\ cell$',
    '5,1': r'$Immune\ cell$',
    '5,2': r'$Immune\ cell$',
    '5,3': r'$Immune\ cell$',
    '6': r'$Immune\ cell$',
    '7': r'$Immune\ cell$',
    '8': r'$Immune\ cell$',
    '9': r'$Immune\ cell$',
    '10': r'$Artefact$',
    '11': r'$Immune\ cell$',
    '12': r'$Immune\ cell$',
    '13': r'$Immune\ cell$',
    '14': r'$Artefact$', 
    '15': r'$Immune\ cell$',
    '16': r'$Immune\ cell$',
    '17,0': r'$Immune\ cell$',
    '17,1': r'$Immune\ cell$',
    '17,2': r'$Immune\ cell$',
    '17,3': r'$Immune\ cell$',
    '18': r'$Artefact$',
    '19': r'$Artefact$',
    '20': r'$Artefact$', 
    '21': r'$Artefact$', 
    '22': r'$Immune\ cell$', 
    '23': r'$Immune\ cell$',
    '24': r'$Immune\ cell$', 
    '25': r'$Immune\ cell$', 
    '26': r'$Artefact$', 
    '27': r'$Immune\ cell$', 
    '28': r'$Immune\ cell$', 
    '29': r'$Immune\ cell$', 
    '30,0': r'$Immune\ cell$', 
    '30,1': r'$Immune\ cell$', 
    '30,2': r'$Immune\ cell$', 
    '30,3': r'$Immune\ cell$'

}

In [ ]:
# Mapper 
celltype_domain = pd.DataFrame({'celltype_domain': adata_0.obs['leiden'].map(lambda x: celltype_domain.get(x, x)).astype('category')})
adata_0.obs = adata_0.obs.merge(celltype_domain, left_index=True, right_index=True, how='left')

In [ ]:
adata_0.obs['celltype_domain'] = adata_0.obs['celltype_domain'].cat.reorder_categories([
    
    r'$Immune\ cell$', 
    r'$Artefact$'
    
])

In [ ]:
sc.pl.umap(adata_0, color=['leiden', 'celltype_leiden', 'celltype_low', 'celltype_high', 'celltype_domain', 'msCC_label', 'log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_MT', 'pct_counts_RP'], frameon=False, ncols=5, wspace=0.1, size=20, legend_loc='on data', use_raw=False)

# Skin subset

In [ ]:
adata_1 = adata[adata.obs.hto_demux_class=='Skin']
adata_1 = adata_1[:, (adata_1.X>=3).sum(axis=0)>=3]

In [ ]:
adata_1 = adata_1.copy()

## SCVI and Dim reduction 

In [ ]:
cache_scvi = True

In [ ]:
if not cache_scvi: 
    
    scvi.model.SCVI.setup_anndata(
    
        adata_1,
        batch_key='sample_name', 
        continuous_covariate_keys=['S_ucell', 'G2M_ucell', 'pct_counts_MT']
        
    )
    
    model = scvi.model.SCVI(
    
        adata_1, 
        n_latent=30, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'

    )
    
    max_epochs = int(np.min([round((20000 / adata_1.n_obs) * 400), 400]))
    
    model.train(max_epochs=max_epochs, check_val_every_n_epoch=1)
    
    model.save('data/object/scvi/annotation/model_1/', overwrite=True)
    
else: 
    
    model = scvi.model.SCVI.load('data/object/scvi/annotation/model_1/', adata=adata_1)

In [ ]:
cache_dim = True

In [ ]:
if not cache_dim: 
    
    adata_1.obsm['latent'] = model.get_latent_representation()
    
    sc.pp.neighbors(adata_1, n_neighbors=30, use_rep='latent')
    sc.tl.leiden(adata_1, resolution=1, flavor='igraph', n_iterations=2, key_added='leiden_annotation')
    sc.tl.umap(adata_1, min_dist=1)
    
    adata_1.write_h5ad('data/object/scvi/annotation/adata_1.h5ad')
    
else: 
    
    adata_1 = sc.read_h5ad('data/object/scvi/annotation/adata_1.h5ad')

In [ ]:
adata_1 = adata_1.raw.to_adata()
adata_1.raw = adata_1.copy()

In [ ]:
sc.pp.normalize_total(adata_1, target_sum=10000)
sc.pp.log1p(adata_1)

In [ ]:
sc.tl.leiden(adata_1, resolution=1.00, restrict_to=('leiden', ['0']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')
sc.tl.leiden(adata_1, resolution=0.50, restrict_to=('leiden_restrict', ['1']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')
sc.tl.leiden(adata_1, resolution=0.50, restrict_to=('leiden_restrict', ['21']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')
sc.tl.leiden(adata_1, resolution=0.25, restrict_to=('leiden_restrict', ['18']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')
sc.tl.leiden(adata_1, resolution=0.25, restrict_to=('leiden_restrict', ['33']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')

adata_1.obs['leiden'] = adata_1.obs['leiden_restrict']
del adata_1.obs['leiden_restrict']

In [ ]:
dp = sc.pl.dotplot(adata_1, celltype_genes_dict, groupby='leiden', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
celltype_leiden = {
    
    '0,0': r'$T_{h}\ RM\ (0,0)$',
    '0,1': r'$T_{c}\ RM\ (0,1)$',
    '0,2': r'$T_{c}\ RM\ (0,2)$',
    '0,3': r'$T_{c}\ RM\ (0,3)$',
    '0,4': r'$T_{c}\ RM\ (0,4)$',
    '0,5': r'$T_{c}\ RM\ (0,5)$',
    '0,6': r'$T_{c}\ RM\ (0,6)$',
    '0,7': r'$T_{c}\ RM\ (0,7)$',
    '0,8': r'$T_{c}\ RM\ (0,8)$',
    '0,9': r'$T_{inv} (0,9)$',
    '1,0': r'$T_{h}\ RM\ (1,0)$',
    '1,1': r'$T_{c}\ RM\ (1,1)$',
    '1,2': r'$T_{c}\ RM\ (1,2)$',
    '1,3': r'$T_{c}\ RM\ (1,3)$',
    '1,4': r'$T_{h}\ RM\ (1,4)$',
    '2': r'$Artefact\ (2)$',
    '3': r'$KC_{sb}\ (3)$',
    '4': r'$KC_{b}\ (4)$',
    '5': r'$F2 (5)$',
    '6': r'$F3 (6)$',
    '7': r'$F3 (7)$',
    '8': r'$F4 (8)$',
    '9': r'$F2 (9)$',
    '10': r'$F1 (10)$',
    '11': r'$F2 (11)$',
    '12': r'$Artefact\ (12)$',
    '13': r'$Artefact\ (13)$',
    '14': r'$Artefact\ (14)$',
    '15': r'$vEC (15)$',
    '16': r'$M\phi\ (16)$',
    '17': r'$M\phi\ (17)$',
    '18,0': r'$cDC2 (18,0)$',
    '18,1': r'$cDC2 (18,1)$',
    '18,2': r'$LC (18,2)$',
    '19': r'$cDC_{Mig}\ (19)$',
    '20': r'$Artefact\ (20)$',
    '21,0': r'$M\phi\ (21,0)$',
    '21,1': r'$M\phi\ (21,1)$',
    '21,2': r'$M\phi\ (21,2)$',
    '21,3': r'$M\phi\ (21,3)$',
    '21,4': r'$LC\ (21,4)$',
    '22': r'$Artefact\ (22)$',
    '23': r'$Pc1 (23)$',
    '24': r'$T_{reg}\ (24)$',
    '25': r'$Artefact\ (25)$',
    '26': r'$F4 (26)$',
    '27': r'$Artefact\ (27)$',
    '28': r'$Artefact\ (28)$',
    '29': r'$lEC (29)$',
    '30': r'$F3 (30)$',
    '31': r'$F2 (31)$',
    '32': r'$Mlc\ (32)$',
    '33,0': r'$Artefact\ (33,0)$',
    '33,1': r'$Mast\ cell\ (33,1)$',
    '34': r'$NK\ CD16-\ (34)$',
    '35': r'$Artefact\ (35)$',
    '36': r'$F1 (36)$',
    '37': r'$Artefact\ (37)$',
    '38': r'$Pc2 (38)$',
    '39': r'$pDC (39)$',
    '40': r'$Artefact\ (40)$',
    '41': r'$cDC1 (41)$',
    '42': r'$Artefact\ (42)$'
    
}

In [ ]:
# Mapper 
celltype_leiden = pd.DataFrame({'celltype_leiden': adata_1.obs['leiden'].map(lambda x: celltype_leiden.get(x, x)).astype('category')})
adata_1.obs = adata_1.obs.merge(celltype_leiden, left_index=True, right_index=True, how='left')

In [ ]:
adata_1.obs['celltype_leiden'] = adata_1.obs['celltype_leiden'].cat.reorder_categories([

    r'$F1 (10)$',
    r'$F1 (36)$',

    r'$F2 (5)$',
    r'$F2 (9)$',
    r'$F2 (11)$',
    r'$F2 (31)$',

    r'$F3 (6)$',
    r'$F3 (7)$',
    r'$F3 (30)$',

    r'$F4 (8)$',
    r'$F4 (26)$',
    
    r'$KC_{b}\ (4)$',
    r'$KC_{sb}\ (3)$',

    r'$vEC (15)$',
    r'$lEC (29)$',

    r'$Pc1 (23)$',
    r'$Pc2 (38)$',

    r'$Mlc\ (32)$',

    r'$T_{h}\ RM\ (0,0)$',
    r'$T_{h}\ RM\ (1,0)$',
    r'$T_{h}\ RM\ (1,4)$',

    r'$T_{reg}\ (24)$',

    r'$T_{c}\ RM\ (0,1)$',
    r'$T_{c}\ RM\ (0,2)$',
    r'$T_{c}\ RM\ (0,3)$',
    r'$T_{c}\ RM\ (0,4)$',
    r'$T_{c}\ RM\ (0,5)$',
    r'$T_{c}\ RM\ (0,6)$',
    r'$T_{c}\ RM\ (0,7)$',
    r'$T_{c}\ RM\ (0,8)$',
    r'$T_{c}\ RM\ (1,1)$',
    r'$T_{c}\ RM\ (1,2)$',
    r'$T_{c}\ RM\ (1,3)$',

    r'$T_{inv} (0,9)$',

    r'$NK\ CD16-\ (34)$',

    r'$M\phi\ (16)$',
    r'$M\phi\ (17)$',
    r'$M\phi\ (21,0)$',
    r'$M\phi\ (21,1)$',
    r'$M\phi\ (21,2)$',
    r'$M\phi\ (21,3)$',

    r'$cDC1 (41)$',
    r'$cDC2 (18,0)$',
    r'$cDC2 (18,1)$',
    r'$LC\ (21,4)$',
    r'$LC (18,2)$',
    r'$cDC_{Mig}\ (19)$',
    r'$pDC (39)$',

    
    r'$Mast\ cell\ (33,1)$',
    
    r'$Artefact\ (2)$',    
    r'$Artefact\ (12)$',
    r'$Artefact\ (13)$',
    r'$Artefact\ (14)$',
    r'$Artefact\ (20)$',
    r'$Artefact\ (22)$',
    r'$Artefact\ (25)$',
    r'$Artefact\ (27)$',
    r'$Artefact\ (28)$',
    r'$Artefact\ (33,0)$',
    r'$Artefact\ (35)$',
    r'$Artefact\ (37)$',
    r'$Artefact\ (40)$',
    r'$Artefact\ (42)$'
    
])

In [ ]:
dp = sc.pl.dotplot(adata_1, celltype_genes_dict, groupby='celltype_leiden', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
celltype_low = {
    
    '0,0': r'$T_{h}\ RM$',
    '0,1': r'$T_{c}\ RM$',
    '0,2': r'$T_{c}\ RM$',
    '0,3': r'$T_{c}\ RM$',
    '0,4': r'$T_{c}\ RM$',
    '0,5': r'$T_{c}\ RM$',
    '0,6': r'$T_{c}\ RM$',
    '0,7': r'$T_{c}\ RM$',
    '0,8': r'$T_{c}\ RM$',
    '0,9': r'$T_{inv}$',
    '1,0': r'$T_{h}\ RM$',
    '1,1': r'$T_{c}\ RM$',
    '1,2': r'$T_{c}\ RM$',
    '1,3': r'$T_{c}\ RM$',
    '1,4': r'$T_{h}\ RM$',
    '2': r'$Artefact$',
    '3': r'$KC_{sb}$',
    '4': r'$KC_{b}$',
    '5': r'$F2$',
    '6': r'$F3$',
    '7': r'$F3$',
    '8': r'$F4$',
    '9': r'$F2$',
    '10': r'$F1$',
    '11': r'$F2$',
    '12': r'$Artefact$',
    '13': r'$Artefact$',
    '14': r'$Artefact$',
    '15': r'$vEC$',
    '16': r'$M\phi$',
    '17': r'$M\phi$',
    '18,0': r'$cDC2$',
    '18,1': r'$cDC2$',
    '18,2': r'$LC$',
    '19': r'$cDC_{Mig}$',
    '20': r'$Artefact$',
    '21,0': r'$M\phi$',
    '21,1': r'$M\phi$',
    '21,2': r'$M\phi$',
    '21,3': r'$M\phi$',
    '21,4': r'$LC$',
    '22': r'$Artefact$',
    '23': r'$Pc1$',
    '24': r'$T_{reg}$',
    '25': r'$Artefact$',
    '26': r'$F4$',
    '27': r'$Artefact$',
    '28': r'$Artefact$',
    '29': r'$lEC$',
    '30': r'$F3$',
    '31': r'$F2$',
    '32': r'$Mlc$',
    '33,0': r'$Artefact$',
    '33,1': r'$Mast\ cell$',
    '34': r'$NK\ CD16-$',
    '35': r'$Artefact$',
    '36': r'$F1$',
    '37': r'$Artefact$',
    '38': r'$Pc2$',
    '39': r'$pDC$',
    '40': r'$Artefact$',
    '41': r'$cDC1$',
    '42': r'$Artefact$'
    
}

In [ ]:
# Mapper 
celltype_low = pd.DataFrame({'celltype_low': adata_1.obs['leiden'].map(lambda x: celltype_low.get(x, x)).astype('category')})
adata_1.obs = adata_1.obs.merge(celltype_low, left_index=True, right_index=True, how='left')

In [ ]:
adata_1.obs['celltype_low'] = adata_1.obs['celltype_low'].cat.reorder_categories([

    r'$F1$',
    r'$F2$',
    r'$F3$',
    r'$F4$',
    r'$KC_{b}$',
    r'$KC_{sb}$',
    r'$vEC$',
    r'$lEC$',
    r'$Pc1$',
    r'$Pc2$',
    r'$Mlc$',
    r'$T_{h}\ RM$',
    r'$T_{reg}$',
    r'$T_{c}\ RM$',
    r'$T_{inv}$',
    r'$NK\ CD16-$',
    r'$M\phi$',
    r'$cDC1$',
    r'$cDC2$',
    r'$LC$',
    r'$cDC_{Mig}$',
    r'$pDC$',
    r'$Mast\ cell$',
    r'$Artefact$'

])

In [ ]:
dp = sc.pl.dotplot(adata_1, celltype_genes_dict, groupby='celltype_low', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
celltype_high = {
    
    '0,0': r'$T\ cell$',
    '0,1': r'$T\ cell$',
    '0,2': r'$T\ cell$',
    '0,3': r'$T\ cell$',
    '0,4': r'$T\ cell$',
    '0,5': r'$T\ cell$',
    '0,6': r'$T\ cell$',
    '0,7': r'$T\ cell$',
    '0,8': r'$T\ cell$',
    '0,9': r'$T\ cell$',
    '1,0': r'$T\ cell$',
    '1,1': r'$T\ cell$',
    '1,2': r'$T\ cell$',
    '1,3': r'$T\ cell$',
    '1,4': r'$T\ cell$',
    '2': r'$Artefact$',
    '3': r'$Epithelial\ cell$',
    '4': r'$Epithelial\ cell$',
    '5': r'$Fibroblast$',
    '6': r'$Fibroblast$',
    '7': r'$Fibroblast$',
    '8': r'$Fibroblast$',
    '9': r'$Fibroblast$',
    '10': r'$Fibroblast$',
    '11': r'$Fibroblast$',
    '12': r'$Artefact$',
    '13': r'$Artefact$',
    '14': r'$Artefact$',
    '15': r'$Endothelial\ cell$',
    '16': r'$Macrophage$',
    '17': r'$Macrophage$',
    '18,0': r'$cDC$',
    '18,1': r'$cDC$',
    '18,2': r'$LC$',
    '19': r'$cDC$',
    '20': r'$Artefact$',
    '21,0': r'$Macrophage$',
    '21,1': r'$Macrophage$',
    '21,2': r'$Macrophage$',
    '21,3': r'$Macrophage$',
    '21,4': r'$LC$',
    '22': r'$Artefact$',
    '23': r'$Pericytes$',
    '24': r'$T\ cell$',
    '25': r'$Artefact$',
    '26': r'$Fibroblast$',
    '27': r'$Artefact$',
    '28': r'$Artefact$',
    '29': r'$Endothelial\ cell$',
    '30': r'$Fibroblast$',
    '31': r'$Fibroblast$',
    '32': r'$Epithelial\ cell$',
    '33,0': r'$Artefact$',
    '33,1': r'$Mast\ cell$',
    '34': r'$ILC$',
    '35': r'$Artefact$',
    '36': r'$Fibroblast$',
    '37': r'$Artefact$',
    '38': r'$Pericytes$',
    '39': r'$pDC$',
    '40': r'$Artefact$',
    '41': r'$cDC$',
    '42': r'$Artefact$'

    
}

In [ ]:
# Mapper 
celltype_high = pd.DataFrame({'celltype_high': adata_1.obs['leiden'].map(lambda x: celltype_high.get(x, x)).astype('category')})
adata_1.obs = adata_1.obs.merge(celltype_high, left_index=True, right_index=True, how='left')

In [ ]:
adata_1.obs['celltype_high'] = adata_1.obs['celltype_high'].cat.reorder_categories([
    
    r'$Fibroblast$',
    
    r'$Epithelial\ cell$', 

    r'$Endothelial\ cell$',
    
    r'$Pericytes$',
    
    r'$T\ cell$',

    r'$ILC$',
    
    r'$Macrophage$',
    
    r'$cDC$',

    r'$LC$', 
    
    r'$pDC$',
    
    r'$Mast\ cell$', 

    r'$Artefact$'
    
])

In [ ]:
dp = sc.pl.dotplot(adata_1, celltype_genes_dict, groupby='celltype_high', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
celltype_domain = {
    
    '0,0': r'$Immune\ cell$',
    '0,1': r'$Immune\ cell$',
    '0,2': r'$Immune\ cell$',
    '0,3': r'$Immune\ cell$',
    '0,4': r'$Immune\ cell$',
    '0,5': r'$Immune\ cell$',
    '0,6': r'$Immune\ cell$',
    '0,7': r'$Immune\ cell$',
    '0,8': r'$Immune\ cell$',
    '0,9': r'$Immune\ cell$',
    '1,0': r'$Immune\ cell$',
    '1,1': r'$Immune\ cell$',
    '1,2': r'$Immune\ cell$',
    '1,3': r'$Immune\ cell$',
    '1,4': r'$Immune\ cell$',
    '2': r'$Artefact$',
    '3': r'$Non-hematopoetic$',
    '4': r'$Non-hematopoetic$',
    '5': r'$Non-hematopoetic$',
    '6': r'$Non-hematopoetic$',
    '7': r'$Non-hematopoetic$',
    '8': r'$Non-hematopoetic$',
    '9': r'$Non-hematopoetic$',
    '10': r'$Non-hematopoetic$',
    '11': r'$Non-hematopoetic$',
    '12': r'$Artefact$',
    '13': r'$Artefact$',
    '14': r'$Artefact$',
    '15': r'$Non-hematopoetic$',
    '16': r'$Immune\ cell$',
    '17': r'$Immune\ cell$',
    '18,0': r'$Immune\ cell$',
    '18,1': r'$Immune\ cell$',
    '18,2': r'$Immune\ cell$',
    '19': r'$Immune\ cell$',
    '20': r'$Artefact$',
    '21,0': r'$Immune\ cell$',
    '21,1': r'$Immune\ cell$',
    '21,2': r'$Immune\ cell$',
    '21,3': r'$Immune\ cell$',
    '21,4': r'$Immune\ cell$',
    '22': r'$Artefact$',
    '23': r'$Non-hematopoetic$',
    '24': r'$Immune\ cell$',
    '25': r'$Artefact$',
    '26': r'$Non-hematopoetic$',
    '27': r'$Artefact$',
    '28': r'$Artefact$',
    '29': r'$Non-hematopoetic$',
    '30': r'$Non-hematopoetic$',
    '31': r'$Non-hematopoetic$',
    '32': r'$Non-hematopoetic$',
    '33,0': r'$Artefact$',
    '33,1': r'$Immune\ cell$',
    '34': r'$Immune\ cell$',
    '35': r'$Artefact$',
    '36': r'$Non-hematopoetic$',
    '37': r'$Artefact$',
    '38': r'$Non-hematopoetic$',
    '39': r'$Immune\ cell$',
    '40': r'$Artefact$',
    '41': r'$Immune\ cell$',
    '42': r'$Artefact$'

}

In [ ]:
# Mapper 
celltype_domain = pd.DataFrame({'celltype_domain': adata_1.obs['leiden'].map(lambda x: celltype_domain.get(x, x)).astype('category')})
adata_1.obs = adata_1.obs.merge(celltype_domain, left_index=True, right_index=True, how='left')

In [ ]:
adata_1.obs['celltype_domain'] = adata_1.obs['celltype_domain'].cat.reorder_categories([
    
    r'$Immune\ cell$',
    
    r'$Non-hematopoetic$',

    r'$Artefact$'
    
])

In [ ]:
sc.pl.umap(adata_1, color=['leiden', 'celltype_leiden', 'celltype_low', 'celltype_high', 'celltype_domain', 'msCC_label', 'log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_MT', 'pct_counts_RP', 'CD4', 'CD8A', 'CD8B', 'CCR6', 'IL7R', 'MAF'], frameon=False, ncols=5, wspace=0.1, size=20, legend_loc='on data', use_raw=False)

# Fetch annotation 

In [ ]:
celltype = pd.concat([adata_0.obs[['celltype_leiden', 'celltype_low', 'celltype_high', 'celltype_domain']], adata_1.obs[['celltype_leiden', 'celltype_low', 'celltype_high', 'celltype_domain']]], axis=0)
adata.obs = adata.obs.merge(celltype, left_index=True, right_index=True, how='left')

In [ ]:
sc.tl.leiden(adata, resolution=0.25, restrict_to=('leiden', ['12']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')
sc.tl.leiden(adata, resolution=0.25, restrict_to=('leiden_restrict', ['13']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')
sc.tl.leiden(adata, resolution=0.25, restrict_to=('leiden_restrict', ['20']), flavor='igraph', n_iterations=2, key_added='leiden_restrict')

adata.obs['leiden'] = adata.obs['leiden_restrict']
del adata.obs['leiden_restrict']

In [ ]:
# Set cells in low quality cluster to Artefact
adata.obs.loc[adata.obs['leiden'].isin(['2', '5', '9', '12,3', '13,1', '14', '16', '20,2', '29', '30']), 'celltype_low'] = r'$Artefact$'
adata.obs.loc[adata.obs['leiden'].isin(['2', '5', '9', '12,3', '13,1', '14', '16', '20,2', '29', '30']), 'celltype_high'] = r'$Artefact$'
adata.obs.loc[adata.obs['leiden'].isin(['2', '5', '9', '12,3', '13,1', '14', '16', '20,2', '29', '30']), 'celltype_domain'] = r'$Artefact$'

In [ ]:
sc.pl.umap(adata, color=['leiden', 'celltype_leiden', 'celltype_low', 'celltype_high', 'celltype_domain', 'msCC_label', 'log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_MT', 'pct_counts_RP'], frameon=False, ncols=5, wspace=0.1, size=20, legend_loc='on data', use_raw=False)

# Save results

In [ ]:
adata.obs[['celltype_leiden', 'celltype_low', 'celltype_high', 'celltype_domain']].to_csv('result/annotation/annotation.csv')